# 155. Min Stack

**Difficulty:** Medium &nbsp;|&nbsp; **Topics:** stack, design, linked-list
&nbsp;|&nbsp; [LeetCode](https://leetcode.com/problems/min-stack/)

Design a stack that supports push, pop, top, and **retrieving the minimum
element** - each in **constant time**.

Implement the `MinStack` class:

- `MinStack()` initializes the stack object.
- `push(val)` pushes the element `val` onto the stack.
- `pop()` removes the element on the top of the stack.
- `top()` gets the top element of the stack.
- `getMin()` retrieves the minimum element in the stack.

You must implement a solution with `O(1)` time complexity for **each**
function.

---

### Example 1

```
Input:  ["MinStack", "push", "push", "push", "getMin", "pop", "top", "getMin"]
        [[],         [-2],   [0],    [-3],   [],       [],    [],    []]
Output: [null,       null,   null,   null,   -3,       null,  0,     -2]

MinStack minStack = new MinStack();
minStack.push(-2);
minStack.push(0);
minStack.push(-3);
minStack.getMin();   // returns -3
minStack.pop();
minStack.top();      // returns 0
minStack.getMin();   // returns -2
```

---

### Constraints

- `-2^31 <= val <= 2^31 - 1`
- Methods `pop`, `top` and `getMin` will always be called on **non-empty**
  stacks.
- At most `3 * 10^4` calls will be made to `push`, `pop`, `top`, and `getMin`.

## Before you write anything

This is your first **design** problem: no algorithm to run over an input, just
a data structure with a promise to keep - four operations, all `O(1)`. And you
have already built the hard part: the `Stack` you wrote for #206 and #114 is a
linked list pushed at the head.

**1.** Start from that Stack. `push`, `pop`, `top` - which of these are
already `O(1)` on your linked-list version? So which single operation is this
whole problem actually about?

**2.** The obvious first idea: keep one extra variable `self.min`, update it
on every push. Now trace this sequence by hand:

```
push(5)   push(3)   push(7)   pop()   pop()
```

Write down what `getMin()` must return after **each** step. At which `pop` does
the single variable give the wrong answer - and what would you have to do to
fix it on the spot? What does that cost? (`O(?)` - this is why the problem is
not trivial.)

**3.** Here is the turn. When you pop the current minimum, the answer you
suddenly need is *the minimum of everything below it* - which is exactly the
value `self.min` held **just before that element was pushed**. So the
information was in your hands at push time, and you threw it away. Where could
you have kept it, so that `pop` gets it back for free?

**4.** You own the `Node` class - you have been adding fields to your own
nodes since the zigzag notebook. What one field do you add to `Node` so that
**every** element remembers the state of the world when it arrived? Once it is
there: what does `getMin()` read, and what does `pop()` have to update?

**5.** The classic trap, for route B below: if you keep a *second* stack that
only records new minimums, and you push onto it when `val < current min` -
trace `push(1)  push(1)  pop()  getMin()`. What goes wrong, and which single
character fixes it?

## Two routes - A is yours, B is the refinement

**A - every node carries its min** *(your answer to question 4)*
Your linked-list Stack, with one extra field per node: the minimum of the
stack *up to and including this node*, computed at push time as
`min(val, current min)`. `getMin` reads the head's field. `pop` just moves the
head - the previous min is sitting in the next node, nothing to recompute.
All four ops `O(1)`, space `O(n)` extra (one int per node).

**B - a second, lazier stack** *(less memory in the common case)*
Keep the main stack plain, plus a `min_stack` that you push onto **only when
the new value is a new minimum** (`val <= getMin()` - question 5 says why
`<=`). On `pop`, if the popped value equals the top of `min_stack`, pop that
too. `getMin` reads `min_stack`'s top. Same `O(1)` everywhere; the min stack
stays tiny when pushes arrive in random order. Which push *sequence* makes
route B's memory exactly as bad as route A's?

Do A first with your own linked list - it is three small edits to code you
have already written. Then B, with plain Python lists if you like
(`append` / `pop` / `[-1]`), which is also the version to submit on LeetCode.

In [4]:
from collections import deque

class Node:
    def __init__(self, val, next=None):
        self.val = val
        self.next = next

class Stack:
    def __init__(self):
        self.head = None
    def append(self,node:Node):
        if self.head is None:
            self.head = node
        else:
            node.next = self.head
            self.head = node
    def pop(self):
        if self.head :
            self.head = self.head.next


class MinStack:

    def __init__(self):
        self.stack = Stack()
        self.min = Stack()

    def push(self, val: int) -> None:
        v1 = Node(val)
        v2 = Node(val)
        if self.min.head and val <= self.min.head.val:
            self.min.append(v1)
        if self.min.head is None :
            self.min.append(v1)
        self.stack.append(v2)



    def pop(self) -> None:
        if self.min.head and self.min.head.val == self.stack.head.val:
            self.min.pop()
        self.stack.pop()

    def top(self) -> int:
        if self.stack.head : return self.stack.head.val
        return 0


    def getMin(self) -> int:
        if self.min.head:  return self.min.head.val
        return 0


### The test runner

Design problems are tested LeetCode-style: a list of operation names and a
list of argument lists, replayed in order against your class. `run` does the
replay. Run this cell; don't edit it.

In [5]:
def run(ops, args):
    """Replay LeetCode-style (ops, args) against MinStack, collect outputs."""
    out, ms = [], None
    for op, a in zip(ops, args):
        if op == "MinStack":
            ms = MinStack()
            out.append(None)
        else:
            out.append(getattr(ms, op)(*a))
    return out


In [6]:
# tests
TESTS = [
    # the LeetCode example
    (["MinStack","push","push","push","getMin","pop","top","getMin"],
     [[],[-2],[0],[-3],[],[],[],[]],
     [None,None,None,None,-3,None,0,-2]),

    # question 2's trace: pop the min, then pop again
    (["MinStack","push","push","push","getMin","pop","getMin","pop","getMin"],
     [[],[5],[3],[7],[],[],[],[],[]],
     [None,None,None,None,3,None,3,None,5]),

    # question 5's trap: duplicate minimums
    (["MinStack","push","push","push","getMin","pop","getMin","pop","getMin"],
     [[],[2],[1],[1],[],[],[],[],[]],
     [None,None,None,None,1,None,1,None,2]),

    # every push is a new min - then unwind it all the way back up
    (["MinStack","push","push","push","push","getMin","pop","getMin","pop","getMin","pop","getMin"],
     [[],[4],[3],[2],[1],[],[],[],[],[],[],[]],
     [None,None,None,None,None,1,None,2,None,3,None,4]),

    # min at the bottom the whole time - pops must NOT disturb it
    (["MinStack","push","push","push","getMin","pop","pop","getMin","top"],
     [[],[1],[5],[3],[],[],[],[],[]],
     [None,None,None,None,1,None,None,1,1]),

    # the constraint boundaries
    (["MinStack","push","push","getMin","top","pop","getMin"],
     [[],[2147483647],[-2147483648],[],[],[],[]],
     [None,None,None,-2147483648,-2147483648,None,2147483647]),
]

for ops, args, expected in TESTS:
    got = run(ops, args)
    print(f"{'OK  ' if got == expected else 'FAIL'} {got}")
    if got != expected:
        print(f"     want {expected}")


OK   [None, None, None, None, -3, None, 0, -2]
OK   [None, None, None, None, 3, None, 3, None, 5]
OK   [None, None, None, None, 1, None, 1, None, 2]
OK   [None, None, None, None, None, 1, None, 2, None, 3, None, 4]
OK   [None, None, None, None, 1, None, None, 1, 1]
OK   [None, None, None, -2147483648, -2147483648, None, 2147483647]


## After it passes

- **Name the idea, it travels.** Route A stores, with each element, a running
  *summary of everything below it* - so undoing a push automatically restores
  the previous summary. Swap `min` for `max`, or for a running *sum*, and the
  same three edits give you MaxStack or a stack with `O(1)` `getSum`. What
  property must an aggregate have for this trick to work in `O(1)` per push?
- **Answer route B's memory question** from the routes cell: which push order
  is its worst case, and how big does `min_stack` get then? One line.
- **Why no `min(self.items)`?** It is one call, it always gives the right
  answer - and it silently breaks the contract. Where else has an `O(n)` thing
  hidden inside an innocent-looking call bitten you already? (#103's
  `insert(0, x)`, #105's `.index()` - collect the family.)
- The siblings on the stack list: #225 Implement Stack using Queues and #232
  Implement Queue using Stacks - both are the same game of keeping a promise
  (`O(1)` where it looks impossible) by paying somewhere else.